# Dashboard: where is the gap largest right now?
Gap ranking, calendar of the next events and alerts feed, built from the live data and the calls in the reports.
The same tables feed the web dashboard; `dashboard.html` is a static export.

In [2]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from altinfo import gap, satellite, earnings, scoring, charts, valuation, eventbeta as eb
from altinfo.charts import NAVY, BLUE, RED, GREY
charts.style()
RAW = ROOT / "data" / "raw"
SNAP = json.load(open(ROOT / "data" / "snapshot_2026-11-09.json"))
LIVE = (RAW / "prices.csv").exists()
print("Mode:", "LIVE (data/raw)" if LIVE else "SNAPSHOT (figures of 9 Nov 2026; run scripts/fetch_data.py for live data)")
def kalshi_live(series):
    """Open Kalshi markets of a series with a mid price (bid/ask midpoint, else last trade)."""
    km = pd.read_csv(RAW / "kalshi_markets.csv")
    m = km[km["series"] == series].copy()
    mid = (m["yes_bid_dollars"] + m["yes_ask_dollars"]) / 2
    tight = (m["yes_ask_dollars"] - m["yes_bid_dollars"]).le(0.10) & m["yes_ask_dollars"].gt(0)
    m["mid"] = mid.where(tight, m["last_price_dollars"])
    return m
def poly_live(pattern):
    pm = pd.read_csv(RAW / "polymarket_markets.csv")
    return pm[pm.question.str.contains(pattern, case=False, na=False, regex=True) & (pm.closed != True)].sort_values("volume", ascending=False)
def live_calls(today="2026-11-10"):
    """The calls registered in the reports, with the market probability updated to the latest live data."""
    calls = pd.DataFrame(SNAP["calls"]).assign(asset=["NVDA", "BRENT"], date=["2026-11-17", "2026-11-30"],
                                                confidence="medium", date_asof=pd.Timestamp(today))
    calls["market_week_ago"] = [SNAP["nvda"]["real"]["polymarket_beat_history"]["2026-11-03"], np.nan]
    if LIVE:
        cd = pd.read_csv(RAW / "kalshi_candles.csv", parse_dates=["date"])
        cd = cd[cd.ticker.str.startswith("KXBRENTMON-26NOV30")].copy()
        cd["strike"] = cd.ticker.str.extract(r"-T([\d.]+)$")[0].astype(float)
        daily = cd.dropna(subset=["price"]).groupby("date").apply(lambda d: gap.ladder_prob_above(d.strike, d.price, 96), include_groups=False)
        bm = kalshi_live("KXBRENTMON").dropna(subset=["floor_strike", "mid"])
        calls.loc[calls.asset == "BRENT", "market"] = gap.ladder_prob_above(bm.floor_strike, bm.mid, 96)
        calls.loc[calls.asset == "BRENT", "market_week_ago"] = daily.loc[:pd.Timestamp(today) - pd.Timedelta(days=7)].iloc[-1]
        nb_ = poly_live(r"N[Vv][Ii][Dd][Ii][Aa].*beat|NVDA.*beat")
        if len(nb_):
            calls.loc[calls.asset == "NVDA", "market"] = float(nb_["price"].iloc[0])
    return calls
def load_prices():
    p = pd.read_csv(RAW / "prices.csv")
    p["date"] = pd.to_datetime(p["date"], format="mixed").dt.normalize()
    return p.pivot_table(index="date", columns="ticker", values="close").sort_index()

Mode: LIVE (data/raw)


In [3]:
from altinfo import dashboard
today = pd.Timestamp("2026-11-10")
calls = live_calls(today)
rank = dashboard.gap_ranking(calls)
view = rank[["asset", "event", "date", "ours", "market", "gap_pts", "label", "confidence", "change_1w_pts"]].copy()
for c in ["ours", "market"]:
    view[c] = view[c].map("{:.0%}".format)
for c in ["gap_pts", "change_1w_pts"]:
    view[c] = view[c].map("{:+.0f}".format)
view.columns = ["Asset", "Event", "Date", "Our view", "Market", "Gap, pts", "Label", "Confidence", "Market 1w, pts"]
view

Asset,Event,Date,Our view,Market,"Gap, pts",Label,Confidence,"Market 1w, pts"
BRENT,Brent above $95.99 on 30 Nov 2026,2026-11-30,62%,26%,+36,UNDER-PRICED,medium,-37
NVDA,NVDA Q3 FY27 EPS above consensus (17 Nov 2026),2026-11-17,90%,94%,-4,OVER-PRICED,medium,+4


**How to read it.** The gap ranking that heads the dashboard: for each live call, our probability, the market's, the gap in points and how much the market moved in the last week. The biggest gaps come first.

In [4]:
cal = dashboard.upcoming(today, days=14, live_calls=list(calls.asset))
cal.assign(date=cal.date.dt.strftime("%a %d %b"))[["date", "event", "names", "group"]].rename(
    columns={"date": "Date", "event": "Event", "names": "Names", "group": "Sector"})

Date,Event,Names,Sector
Thu 12 Nov,EIA weekly stocks,"Brent crude, WTI crude",Energy
Mon 16 Nov,USDA Crop Progress,"Corn, Wheat, Soybeans",Agriculture
Tue 17 Nov,Q3 FY2027 results (after close),NVIDIA,Semiconductors
Mon 23 Nov,USDA Crop Progress,"Corn, Wheat, Soybeans",Agriculture
Sun 29 Nov,OPEC+ meeting,Brent crude,Energy
Mon 30 Nov,Kalshi month-end settlement,Brent crude,Energy
Tue 01 Dec,GTC Washington keynote,NVIDIA,Semiconductors


**How to read it.** Key dates in the next two weeks for every company and commodity we cover, plus the later dates of the two live calls: results, official inventory and crop reports, OPEC+ meetings and settlement of prediction market contracts. The same report for several names (for example USDA Crop Progress) is one row.

In [5]:
jumps = None
if LIVE:
    ph = pd.read_csv(RAW / "polymarket_history.csv", parse_dates=["date"])
    ph = ph[ph.key.fillna("") != ""]
    from altinfo.monitors import market_jumps
    jumps = market_jumps(ph[ph.date >= today - pd.Timedelta(days=60)], threshold_pts=10)
    px = load_prices()
else:
    px = None
alerts = dashboard.alerts_feed(rank, jumps, px, since=today - pd.Timedelta(days=30))
alerts.head(15)

Date,Type,Asset,Text
2026-11-10,Gap,Brent crude,"Brent above $95.99 on 30 Nov 2026: gap +36 pts, under-priced"
2026-11-05,Price,NVIDIA,closed -9.3% on the day (NVDA)
2026-11-04,Price,Brent crude,closed -8.7% on the day (BZ=F)
2026-11-04,Price,WTI crude,closed -8.3% on the day (CL=F)
2026-10-30,Price,NVIDIA,closed -4.2% on the day (NVDA)
2026-10-22,Price,NVIDIA,closed -4.4% on the day (NVDA)
2026-10-21,Price,WTI crude,closed -4.6% on the day (CL=F)


**How to read it.** The alerts a client would have received in the last month: a gap above 5 points, a prediction market moving 10 points in a day, or a price moving more than 4% in a day.

In [6]:
sparks = {}
km = pd.read_csv(RAW / "kalshi_markets.csv") if LIVE else None
pm = pd.read_csv(RAW / "polymarket_markets.csv") if LIVE else None
bh = SNAP["nvda"]["real"]["polymarket_beat_history"]
sparks["NVDA"] = [v * 100 for v in bh.values()]
if LIVE:
    cd = pd.read_csv(RAW / "kalshi_candles.csv", parse_dates=["date"])
    cd = cd[cd.ticker.str.startswith("KXBRENTMON-26NOV30")].copy()
    cd["strike"] = cd.ticker.str.extract(r"-T([\d.]+)$")[0].astype(float)
    daily = cd.dropna(subset=["price"]).groupby("date").apply(lambda d: gap.ladder_prob_above(d.strike, d.price, 96), include_groups=False)
    sparks["BRENT"] = list(daily.values * 100)
cover = dashboard.coverage_table(px, km, pm, calls, today) if px is not None else None
from altinfo.config import HOST, PORT
APP = f"http://{HOST}:{PORT}/"   # the live search app (python server/app.py)
NAVD = [("Search", APP), ("Track record", APP + "track"), ("Event betas", "notebooks/03_event_betas.html"),
        ("Dashboard", APP + "dashboard"), ("Monitors", "notebooks/07_monitors.html")]
from altinfo.coverage import COVERAGE
links = {k: f"signals/{k}.html" for k in COVERAGE}
html = dashboard.to_html(rank, cal, alerts, cover, sparks, asof=f"{today:%d %B %Y}", nav=NAVD, signal_links=links)
(ROOT / "dashboard.html").write_text(html)
print("written dashboard.html")

written dashboard.html


**How to read it.** Writes the dashboard page (dashboard.html) from the tables above. The page has filters by sector, region and size of the gap, and each name links to its signal page.

## Signal pages
One page per covered name with the history of each index, how the prediction market odds moved, attention on X and
the supply chain map with live alerts, linked from the dashboard (click a name).

In [8]:
from altinfo import signals
(ROOT / "signals").mkdir(exist_ok=True)
REPORTS = {"NVDA": "../notebooks/01_nvda_preview.html", "BRENT": "../notebooks/02_brent_preview.html"}
NAVS = [(l, u if u.startswith("http") else "../" + u) for l, u in NAVD]
rows = []
for k in COVERAGE:
    sig = signals.collect(k, ROOT, today)
    (ROOT / "signals" / f"{k}.html").write_text(signals.page_html(k, sig, NAVS, REPORTS.get(k), f"{today:%d %B %Y}"))
    rows.append({"Name": COVERAGE[k]["name"], "Charts": len(sig["charts"]), "Places on the map": len(COVERAGE[k]["sites"]),
                 "Live alerts": len(sig["alerts"]), "Page": f"signals/{k}.html"})
pd.DataFrame(rows)

Name,Charts,Places on the map,Live alerts,Page
NVIDIA,5,10,0,signals/NVDA.html
Brent crude,4,7,0,signals/BRENT.html
WTI crude,2,4,0,signals/WTI.html
Copper,1,12,1,signals/COPPER.html
Gold,1,6,0,signals/GOLD.html
Corn,1,5,0,signals/CORN.html
Wheat,1,5,0,signals/WHEAT.html
Soybeans,1,4,0,signals/SOYBEANS.html
Coffee,1,4,0,signals/COFFEE.html
Sugar,1,4,0,signals/SUGAR.html


**How to read it.** One signal page per covered name: index histories, prediction market odds, attention and the supply chain map. The table counts what each page shows; live alerts are earthquakes within 100 km or fires within 30 km of a place.

## Free and Institutional tiers
What each tier sees. The free tier never shows live satellite readings; reports are released to it after each event.

In [10]:
from altinfo import tiers
tiers.table()

Feature,Free,Institutional
Earnings and commodity calendar,Yes,Yes
Headline prediction market odds,Yes,Yes
Attention on X,Yes,Yes
Ticker search,"Basic: price, markets, attention","Full: satellite indices, supply chain, gap"
Event reports,Released after each event,"Live, before each event"
Dashboard and gap ranking,No,Yes
Live satellite readings,Never,Yes
Personalized monitors,No,"Yes, private to each client"
Custom sites to track,No,Yes


**How to read it.** What each tier gets. Free users see the calendar, headline odds, attention and a basic search; live satellite readings, the gap ranking and monitors are for institutional clients only.